# External test — classifier-only, no threshold

Simpler variant of `evaluate_external_test.ipynb`: no centroid comparison, no confidence threshold — just the trained classifier's raw prediction on every croppable image, reported as (true class, predicted class, confidence).

**Step 1** crops `data/external_test/<class>/*` with MTCNN and saves only the images where a face was actually found into `data/external_test_cropped/<class>/` (skips + logs the rest). **Step 2** runs the classifier on that cropped set.

## Step 1 — Crop with MTCNN, keep only images with a detected face

In [9]:
import os
import sys
import glob
import json
import numpy as np
import os
from PIL import Image

PROJECT_ROOT = os.path.abspath('..')
sys.path.insert(0, PROJECT_ROOT)  # so "src.face_detector" is importable

from src.face_detector import detect_faces

SRC_DIR = os.path.join(PROJECT_ROOT, 'data', 'external_test')
OUT_DIR = os.path.join(PROJECT_ROOT, 'data', 'external_test_cropped')

# Crops smaller than this on either side are excluded - too little real face
# detail for a fair/meaningful prediction (same spirit as the app's own
# "face crop too small" guard, just tightened here since we can review the
# excluded list interactively instead of silently degrading UX).
MIN_CROP_SIZE = 50

# 5% margin added around MTCNN's tight box on every side - gives a little
# breathing room (chin/forehead/ears) instead of an exact-pixel crop.
MARGIN = 0.05

with open(os.path.join(PROJECT_ROOT, 'models', 'embeddings', 'class_names.json')) as f:
    class_names = json.load(f)


def pick_largest_face(boxes):
    return max(boxes, key=lambda b: b[2] * b[3])


def clear_dir(path):
    if os.path.isdir(path):
        for f in glob.glob(os.path.join(path, '*')):
            os.remove(f)


no_face_log = []
too_small_log = []
total_seen = 0
total_saved = 0

for class_name in class_names:
    class_dir = os.path.join(SRC_DIR, class_name)
    if not os.path.isdir(class_dir):
        print(f"⚠️  No folder for {class_name} - skipping")
        continue

    out_dir = os.path.join(OUT_DIR, class_name)
    os.makedirs(out_dir, exist_ok=True)
    clear_dir(out_dir)  # clean slate each run

    files = sorted(glob.glob(os.path.join(class_dir, '*')))
    saved_this_class = 0
    for f in files:
        total_seen += 1
        try:
            img = Image.open(f).convert('RGB')
        except Exception as e:
            print(f"  could not open {f}: {e}")
            continue

        boxes = detect_faces(np.array(img))
        if len(boxes) == 0:
            no_face_log.append(f)
            continue

        x, y, w, h = pick_largest_face(boxes)
        x, y = max(0, x), max(0, y)

        # 5% margin on each side - gives the face a little breathing room
        # (chin/forehead/ears) instead of the exact tight MTCNN box.
        mx, my = int(round(w * MARGIN)), int(round(h * MARGIN))
        x1 = max(0, x - mx)
        y1 = max(0, y - my)
        x2 = min(img.width, x + w + mx)
        y2 = min(img.height, y + h + my)
        crop_w, crop_h = x2 - x1, y2 - y1

        if crop_w < MIN_CROP_SIZE or crop_h < MIN_CROP_SIZE:
            too_small_log.append((f, crop_w, crop_h))
            continue

        face_crop = img.crop((x1, y1, x2, y2))

        out_path = os.path.join(out_dir, f"{saved_this_class:03d}.png")
        face_crop.save(out_path)
        saved_this_class += 1
        total_saved += 1

    print(f"  {class_name:20s} {saved_this_class}/{len(files)} image(s) had a detectable face")

print(f"\nDone. {total_saved}/{total_seen} images saved to {OUT_DIR}/")
if no_face_log:
    print(f"\n{len(no_face_log)} image(s) had NO detectable face (excluded):")
    for f in no_face_log:
        print(f"  {f}")
if too_small_log:
    print(f"\n{len(too_small_log)} image(s) had a detected face SMALLER than {MIN_CROP_SIZE}x{MIN_CROP_SIZE}px after margin (excluded):")
    for f, w, h in too_small_log:
        print(f"  {w}x{h}px  {f}")

  Arya_Stark           6/7 image(s) had a detectable face
  Brandon_Stark        5/6 image(s) had a detectable face
  Catelyn_Stark        5/6 image(s) had a detectable face
  Cersei_Lannister     5/6 image(s) had a detectable face
  Eddard_Stark         6/6 image(s) had a detectable face
  Jaime_Lannister      7/7 image(s) had a detectable face
  Joffrey_Baratheon    6/7 image(s) had a detectable face
  Jon_Snow             7/7 image(s) had a detectable face
  Maester_Luwin        4/7 image(s) had a detectable face
  Rickon_Stark         1/5 image(s) had a detectable face
  Robb_Stark           4/5 image(s) had a detectable face
  Robert_Baratheon     8/9 image(s) had a detectable face
  Rodrik_Cassel        4/4 image(s) had a detectable face
  Sansa_Stark          4/7 image(s) had a detectable face
  Theon_Greyjoy        7/9 image(s) had a detectable face

Done. 79/98 images saved to c:\Users\itsme\OneDrive\Desktop\CinefaceInsight_Deploy\data\external_test_cropped/

4 image(s) had NO

## Step 2 — Run the classifier only — no centroid, no threshold — report true/predicted/confidence

In [10]:
import cv2
from deepface import DeepFace
import keras

try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

FACENET_MODEL_NAME = 'Facenet512'
CLASSIFIER_PATH = os.path.join(PROJECT_ROOT, 'models', 'facenet_classifier.keras')

classifier = keras.models.load_model(CLASSIFIER_PATH)


def get_embedding(pil_image_rgb):
    img_bgr = cv2.cvtColor(np.array(pil_image_rgb), cv2.COLOR_RGB2BGR)
    result = DeepFace.represent(
        img_path=img_bgr,
        model_name=FACENET_MODEL_NAME,
        detector_backend='skip',  # already cropped to just the face
        enforce_detection=False,
    )
    return np.array(result[0]['embedding'], dtype=np.float32)


rows = []
for true_name in class_names:
    class_dir = os.path.join(OUT_DIR, true_name)
    if not os.path.isdir(class_dir):
        continue
    for f in sorted(glob.glob(os.path.join(class_dir, '*'))):
        img = Image.open(f).convert('RGB')
        embedding = get_embedding(img)
        probs = classifier.predict(embedding[np.newaxis, :], verbose=0)[0]
        pred_idx = int(np.argmax(probs))
        rows.append({
            'true_class': true_name,
            'predicted_class': class_names[pred_idx],
            'confidence': float(probs[pred_idx]),
            'file': f,
        })

print(f"{'true_class':20s} {'predicted_class':20s} {'confidence':>10s}")
for r in rows:
    flag = '' if r['true_class'] == r['predicted_class'] else '  <-- wrong'
    print(f"{r['true_class']:20s} {r['predicted_class']:20s} {r['confidence']:10.4f}{flag}")

correct = sum(1 for r in rows if r['true_class'] == r['predicted_class'])
print(f"\n{correct}/{len(rows)} correct ({100*correct/len(rows):.1f}%) - raw argmax, no threshold applied")

true_class           predicted_class      confidence
Arya_Stark           Arya_Stark               0.5347
Arya_Stark           Arya_Stark               0.9023
Arya_Stark           Arya_Stark               0.7665
Arya_Stark           Arya_Stark               0.8988
Arya_Stark           Arya_Stark               0.5886
Arya_Stark           Arya_Stark               0.8539
Brandon_Stark        Brandon_Stark            0.9114
Brandon_Stark        Brandon_Stark            0.9512
Brandon_Stark        Brandon_Stark            0.3632
Brandon_Stark        Brandon_Stark            0.7858
Brandon_Stark        Brandon_Stark            0.4221
Catelyn_Stark        Catelyn_Stark            0.9371
Catelyn_Stark        Catelyn_Stark            0.7969
Catelyn_Stark        Catelyn_Stark            0.9236
Catelyn_Stark        Jon_Snow                 0.2045  <-- wrong
Catelyn_Stark        Catelyn_Stark            0.5496
Cersei_Lannister     Cersei_Lannister         0.7145
Cersei_Lannister     Cersei_Lannist

In [12]:
threshold=0.1
above = [r for r in rows if r['confidence'] > threshold]
correct_above = sum(1 for r in above if r['true_class'] == r['predicted_class'])

print(f"Predictions with confidence > {100*threshold}%: {len(above)}/{len(rows)} ({100*len(above)/len(rows):.1f}%) ")
print(f"Correct among those:           {correct_above}/{len(above)} ({100*correct_above/len(above):.1f}%)" if above else "Correct among those:0/0")

Predictions with confidence > 10.0%: 79/79 (100.0%) 
Correct among those:           75/79 (94.9%)
